

---
### Purpose of this Notebook:
1. **Environment Setup:** Install `rfdetr[train,loggers]` with ALL training dependencies.
2. **Hyperparameter Configuration:** Define CONFIG cell with epochs (>= 20), batch size, gradient accumulation, and learning rate.
3. **Model Initialization & Training:** Fine-tune the transformer-based RF-DETR Nano model on the YOLO split format.
4. **Visualizations:** Plot training loss and mAP metrics curves over epochs.
5. **Save Best Checkpoint:** Record the best EMA weights path for the final comparative notebook.

> **Why RF-DETR Nano?** Unlike CNN-based YOLO models, RF-DETR uses a DINOv2 Vision Transformer (ViT) backbone
> with deformable attention, allowing it to capture global context — beneficial for fisheye camera distortions
> where passengers appear at varying positions and sizes.


## 1. ENVIRONMENT SETUP
We install `rfdetr` with the `[train,loggers]` extras which include `faster_coco_eval`, `pytorch_lightning`,
and other required training dependencies.

> **Important:** Using `rfdetr[train,loggers]` is mandatory. A plain `pip install rfdetr` installs only
> inference dependencies and will raise `ModuleNotFoundError: No module named faster_coco_eval` during training.


In [1]:
# Install rfdetr with ALL training dependencies
!pip install "rfdetr[train,loggers]" -q

# Verify the critical dependency is present
import faster_coco_eval
print('faster_coco_eval installed successfully ✅')
from rfdetr import RFDETRNano
print('rfdetr imported successfully ✅')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 14.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 100.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 99.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.4 MB/s eta 0:00:00:00:01
   ━━

/usr/local/lib/python3.12/dist-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


## 2. CONFIGURATION CELL
Hyperparameters for training a Transformer-based detector.

| Parameter | Value | Reasoning |
|---|---|---|
| `EPOCHS` | 20 | Assignment minimum for RF-DETR |
| `BATCH_SIZE` | 4 | Fits in T4 16GB VRAM |
| `GRAD_ACCUM_STEPS` | 4 | Effective batch = 4×4 = **16** |
| `INITIAL_LR` | 1e-4 | Transformers need lower LR than CNNs |
| `resolution` | 640 | Matches YOLO input size for fair comparison |


In [2]:
# ================= CONFIGURATION CELL =================
import os

KAGGLE_ENVIRONMENT = True  # Toggle to False if running locally

if KAGGLE_ENVIRONMENT:
    # Point to the YOLO-format dataset (from Notebook 1 output or Kaggle input)
    DATASET_DIR = '/kaggle/input/datasets/mdnahidulislam01/senticar-dataset-yolo-od'
    OUTPUT_DIR  = '/kaggle/working/RFDETR_sentinelcar'
else:
    DATASET_DIR = './SentiCAR_dataset_yolo_OD'
    OUTPUT_DIR  = './RFDETR_sentinelcar'

# Training Hyperparameters
EPOCHS           = 20
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 4   # Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 16
INITIAL_LR       = 1e-4
DEVICE           = 'cuda'  # 'cuda' for GPU, 'cpu' for CPU

print('RF-DETR Nano training hyperparameters configured.')
print(f'  Dataset    : {DATASET_DIR}')
print(f'  Output     : {OUTPUT_DIR}')
print(f'  Epochs     : {EPOCHS}')
print(f'  Batch size : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})')
print(f'  LR         : {INITIAL_LR}')


RF-DETR Nano training hyperparameters configured.
  Dataset    : /kaggle/input/datasets/mdnahidulislam01/senticar-dataset-yolo-od
  Output     : /kaggle/working/RFDETR_sentinelcar
  Epochs     : 20
  Batch size : 4 (effective: 16)
  LR         : 0.0001


## 3. RF-DETR NANO MODEL TRAINING
We initialize the RFDETRNano model and train it for 20 epochs.

The model reads the YOLO-format dataset directory directly.
It expects the following folder structure (same as Notebook 1 output):
```
SentiCAR_dataset_yolo_OD/
├── train/
|   ├── images/
|   └── labels/
├── val/
|   ├── images/
|   └── labels/
└── data.yaml
```


In [3]:
import os
import yaml

# Source: your Kaggle input (read-only)
SOURCE = '/kaggle/input/datasets/mdnahidulislam01/senticar-dataset-yolo-od'

# Destination: writable working directory
DATASET_DIR = '/kaggle/working/SentiCAR_rfdetr'
os.makedirs(DATASET_DIR, exist_ok=True)

splits = {
    'train': 'train',
    'val': 'valid',   # <-- Map 'val' source to 'valid' destination
    'test': 'test'
}

# Symlink train/val/test from input → working (no copying needed!)
for src_split, dst_split in splits.items():
    src = f'{SOURCE}/{src_split}'
    dst = f'{DATASET_DIR}/{dst_split}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked: {dst} → {src}')
    else:
        print(f'Skipped (already exists or source missing): {dst}')

# Write data.yaml into the writable directory
data_yaml_content = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 2,
    'names': {0: 'Person_Active', 1: 'Person_Not_Active'}
}
yaml_path = f'{DATASET_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f'\ndata.yaml created at: {yaml_path}')
print('Dataset structure ready for RF-DETR ✅')

# Verify
for item in os.listdir(DATASET_DIR):
    print(f'  {DATASET_DIR}/{item}')

Linked: /kaggle/working/SentiCAR_rfdetr/train → /kaggle/input/datasets/mdnahidulislam01/senticar-dataset-yolo-od/train
Linked: /kaggle/working/SentiCAR_rfdetr/valid → /kaggle/input/datasets/mdnahidulislam01/senticar-dataset-yolo-od/val
Linked: /kaggle/working/SentiCAR_rfdetr/test → /kaggle/input/datasets/mdnahidulislam01/senticar-dataset-yolo-od/test

data.yaml created at: /kaggle/working/SentiCAR_rfdetr/data.yaml
Dataset structure ready for RF-DETR ✅
  /kaggle/working/SentiCAR_rfdetr/data.yaml
  /kaggle/working/SentiCAR_rfdetr/valid
  /kaggle/working/SentiCAR_rfdetr/train
  /kaggle/working/SentiCAR_rfdetr/test


In [ ]:
# Initialize RF-DETR Nano model with resolution 640
import os
# PyTorch Lightning-কে ফোর্স করা যাতে সে প্রতি ১ স্টেপ পর পর প্রোগ্রেস বার রিফ্রেশ করে
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PL_PROGRESS_BAR_REFRESH_RATE"] = "1"

model = RFDETRNano(resolution=640)

os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASET_DIR = '/kaggle/working/SentiCAR_rfdetr'

print('Starting RF-DETR Nano training...')
model.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=INITIAL_LR,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
)

print('RF-DETR Nano training completed ✅')


[2026-06-29 03:45:22] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-06-29 03:45:22] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-29 03:45:22] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-29 03:45:24] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-06-29 03:45:25] [WARNING] rf-detr - Pretrained weights at '/root/.roboflow/models/rf-detr-nano.pth' loaded only partially — this typically produces lower accuracy. 1 model parameter(s) not in checkpoint (left at random init): [_kp_active_mask]. Check that the model configuration (encoder, hidden_dim, out_feature_indexes, projector_scale, ...) matches the architecture the checkpoint was trained with.
[2026-06-29 03:45:25] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-29 03:45:25] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Starting RF-DETR Nano training...
[2026-06-29 03:45:26] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/RFDETR_sentinelcar exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model       │ LWDETR       │ 30.5 M │ train │     0 │
│ 1 │ criterion   │ SetCriterion │      0 │ train │     0 │
│ 2 │ postprocess │ PostProcess  │      0 │ train │     0 │
└───┴─────────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 30.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 30.5 M                                                                                               
Total estimated model params size (MB): 122.176                                                                    
Modules in train mode: 449                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

[2026-06-29 03:51:59] [INFO] rf-detr - Best EMA mAP improved to 0.0465 (epoch 0)
[2026-06-29 04:57:47] [INFO] rf-detr - Best regular checkpoint saved to /kaggle/working/RFDETR_sentinelcar/checkpoint_best_regular.pth (epoch 0, monitor=val/mAP_50_95, value=0.560996)
[2026-06-29 04:57:47] [INFO] rf-detr - Best EMA mAP improved to 0.5820 (epoch 0)
[2026-06-29 06:03:25] [INFO] rf-detr - Best EMA mAP improved to 0.5840 (epoch 1)


## 4. TRAINING CURVES VISUALIZATION
We read the training logs from the output directory and plot loss and validation mAP curves.


In [ ]:
import matplotlib.pyplot as plt
import glob
import json

log_dirs = glob.glob(os.path.join(OUTPUT_DIR, '**/metrics.json'), recursive=True)
losses  = []
map50s  = []
epochs_log = []

if log_dirs:
    log_path = log_dirs[0]
    with open(log_path, 'r') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                if 'loss' in data and 'epoch' in data:
                    losses.append(data['loss'])
                    epochs_log.append(data['epoch'])
                    map50s.append(data.get('map50', 0.0))
            except:
                continue

if losses:
    fig, axs = plt.subplots(1, 2, figsize=(16, 5))
    axs[0].plot(epochs_log, losses, color='tomato', marker='o', label='Training Loss')
    axs[0].set_title('RF-DETR Nano Training Loss Convergence', fontsize=12, fontweight='bold')
    axs[0].set_xlabel('Epochs')
    axs[0].set_ylabel('Loss')
    axs[0].grid(True, linestyle='--', alpha=0.6)
    axs[0].legend()

    axs[1].plot(epochs_log, map50s, color='seagreen', marker='s', label='Val mAP@50')
    axs[1].set_title('RF-DETR Nano Validation mAP@50', fontsize=12, fontweight='bold')
    axs[1].set_xlabel('Epochs')
    axs[1].set_ylabel('mAP@50')
    axs[1].grid(True, linestyle='--', alpha=0.6)
    axs[1].legend()

    plt.suptitle('RF-DETR Nano — SentinelCAR Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Note: metrics.json not found — training logs may use a different format.')
    print('Check the OUTPUT_DIR for available log files:')
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            print(' ', os.path.join(root, f))


## 5. WEIGHT PRESERVATION
RF-DETR saves two checkpoint files:
- `checkpoint_best_total.pth` — Best overall checkpoint (use this for evaluation)
- `checkpoint_best_ema.pth`   — Best EMA-smoothed checkpoint


In [ ]:
import shutil

rf_best_total = os.path.join(OUTPUT_DIR, 'checkpoint_best_total.pth')
rf_best_ema   = os.path.join(OUTPUT_DIR, 'checkpoint_best_ema.pth')

print('RF-DETR Best Checkpoints:')
print(f'  Best Total : {rf_best_total}  [exists: {os.path.exists(rf_best_total)}]')
print(f'  Best EMA   : {rf_best_ema}   [exists: {os.path.exists(rf_best_ema)}]')

# Zip for easy download
shutil.make_archive('/kaggle/working/rfdetr_results', 'zip', OUTPUT_DIR)
print('ZIP created: /kaggle/working/rfdetr_results.zip — download from right panel.')


## 6. SUMMARY TABLE
Configuration and key metrics for the RF-DETR Nano transformer model.


In [ ]:
import pandas as pd
import IPython.display as display

rf_summary = {
    'Model'             : ['RF-DETR Nano'],
    'Backbone'          : ['DINOv2 (ViT-S/16)'],
    'Epochs Trained'    : [EPOCHS],
    'Learning Rate'     : [INITIAL_LR],
    'Effective Batch'   : [BATCH_SIZE * GRAD_ACCUM_STEPS],
    'Best Val mAP@50'   : [max(map50s) if map50s else 'See training logs'],
    'Checkpoint'        : [rf_best_total]
}

rf_df = pd.DataFrame(rf_summary)
display.display(rf_df)
